In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
import yaml
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go


In [2]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)


In [6]:
df_gb = pd.read_csv(config["input_data"]["file4"])
df_gb.head()

,Realm_Name,Ecoregion_Name,Country_Name,Distance_to_Shore,Exposure,Cyclone_Frequency,Date_Day,Date_Month,Date_Year,Depth_m,...,SSTA_Standard_Deviation,SSTA_Mean,SSTA_DHW,SSTA_DHW_Standard_Deviation,TSA,TSA_Standard_Deviation,Date,Site_Comments,Sample_Comments,Bleaching_Comments
0,Central Indo-Pacific,Central and northern Great Barrier Reef,Australia,57467.04,Sheltered,48.94,9,7,2017,5,...,0.79,0,0.0,2.89,-4.71,2.06,2017-07-09,nd,nd,nd
1,Central Indo-Pacific,Central and northern Great Barrier Reef,Australia,57467.04,Sheltered,48.94,9,7,2017,5,...,0.79,0,0.0,2.89,-4.71,2.06,2017-07-09,nd,nd,nd
2,Central Indo-Pacific,Central and northern Great Barrier Reef,Australia,57656.01,Sheltered,48.94,9,7,2017,2,...,0.79,0,0.0,2.89,-4.71,2.06,2017-07-09,nd,nd,nd
3,Central Indo-Pacific,Central and northern Great Barrier Reef,Australia,57656.01,Sheltered,48.94,9,7,2017,2,...,0.79,0,0.0,2.89,-4.71,2.06,2017-07-09,nd,nd,nd
4,Central Indo-Pacific,Central and northern Great Barrier Reef,Australia,43199.05,Sheltered,43.31,7,7,2017,4,...,0.82,0,0.0,3.07,-4.77,2.20,2017-07-07,nd,nd,nd


In [12]:
print(list(df_gb.columns))


['Realm_Name', 'Ecoregion_Name', 'Country_Name', 'Distance_to_Shore', 'Exposure', 'Cyclone_Frequency', 'Date_Day', 'Date_Month', 'Date_Year', 'Depth_m', 'Substrate_Name', 'Percent_Cover', 'Bleaching_Level', 'Percent_Bleaching', 'ClimSST', 'SSTA', 'SSTA_Standard_Deviation', 'SSTA_Mean', 'SSTA_DHW', 'SSTA_DHW_Standard_Deviation', 'TSA', 'TSA_Standard_Deviation', 'Date', 'Site_Comments', 'Sample_Comments', 'Bleaching_Comments']


In [32]:
#see all values of columns
pd.set_option('display.max_rows', None)
# Reset option to default if needed
#pd.reset_option('display.max_rows')

In [51]:
def clean_gb(df_gb):
    #Convert all column names to lowercase
    df_gb.columns = df_gb.columns.str.lower()

   # Specify columns to drop
    columns_to_drop = [
        'realm_name', 'date_day', 'date_month', 'date_year', 
        'depth_m', 'sample_comments', 'site_comments',  
        'substrate_name', 'percent_cover', 'ssta_mean', 
        'ssta_dhw', 'ssta_dhw_standard_deviation', 
        'tsa', 'tsa_standard_deviation', 'bleaching_comments'
    ]

    # Drop specified columns
    df_gb = df_gb.drop(columns=columns_to_drop, errors='ignore')

    # Remove duplicate rows
    df_gb = df_gb.drop_duplicates()

    # Backfill NaN values in 'percent_bleaching'
    df_gb['percent_bleaching'] = df_gb['percent_bleaching'].ffill()

    # Return the cleaned DataFrame
    return df_gb

# Clean the DataFrame
df_gb_cleaned = clean_gb(df_gb)

# Display the cleaned DataFrame
display(df_gb_cleaned)

,ecoregion_name,country_name,distance_to_shore,exposure,cyclone_frequency,bleaching_level,percent_bleaching,climsst,ssta,ssta_standard_deviation,date
0,Central and northern Great Barrier Reef,Australia,57467.04,Sheltered,48.94,Population,0.00,296.66,0.42,0.79,2017-07-09
2,Central and northern Great Barrier Reef,Australia,57656.01,Sheltered,48.94,Population,0.00,296.66,0.42,0.79,2017-07-09
4,Central and northern Great Barrier Reef,Australia,43199.05,Sheltered,43.31,Population,0.00,296.51,0.77,0.82,2017-07-07
6,Central and northern Great Barrier Reef,Australia,41586.65,Sheltered,48.37,Population,0.00,298.59,-2.30,2.23,2017-08-05
8,Central and northern Great Barrier Reef,Australia,145.07,Exposed,43.52,Population,0.00,295.84,1.62,1.02,2017-07-07
10,Southern Great Barrier Reef,Australia,54.69,Sheltered,51.46,Population,0.25,299.83,0.00,0.72,2015-11-24
12,Southern Great Barrier Reef,Australia,459.75,Sheltered,51.46,Population,0.25,297.41,0.80,0.72,2016-09-22
14,Southern Great Barrier Reef,Australia,184.65,Sheltered,51.57,Population,0.25,297.41,0.80,0.72,2016-09-22
16,Central and northern Great Barrier Reef,Australia,57425.12,Sheltered,48.94,Population,0.50,296.66,0.42,0.79,2017-07-08
18,Central and northern Great Barrier Reef,Australia,220.26,Exposed,43.52,Population,0.50,301.47,1.59,1.02,2017-10-15


In [49]:
df_gb_cleaned.to_csv('global_bleaching_cleaned.csv', index=False)

In [52]:
def sort_by_date(df, date_column):
    # Ensure the date column is in datetime format
    df[date_column] = pd.to_datetime(df[date_column])

    # Sort the DataFrame by the date column
    df_sorted = df.sort_values(by=date_column)

    # Reset the index to reflect the new ordering
    df_sorted.reset_index(drop=True, inplace=True)

    return df_sorted
df_sorted = sort_by_date(df_gb_cleaned, 'date')
display(df_sorted)

,ecoregion_name,country_name,distance_to_shore,exposure,cyclone_frequency,bleaching_level,percent_bleaching,climsst,ssta,ssta_standard_deviation,date
0,Central and northern Great Barrier Reef,Australia,2865.08,Sheltered,43.52,Population,6.50,296.78,1.64,1.02,2015-07-25
1,Central and northern Great Barrier Reef,Australia,2795.93,Sheltered,43.52,Population,8.75,296.78,1.64,1.02,2015-07-25
2,Central and northern Great Barrier Reef,Australia,216.38,Sometimes,43.39,Population,13.25,296.52,1.11,0.98,2015-07-26
3,Central and northern Great Barrier Reef,Australia,326.99,Sheltered,43.39,Population,24.25,296.53,1.11,0.96,2015-07-26
4,Central and northern Great Barrier Reef,Australia,742.91,Sheltered,63.92,Population,3.00,297.64,0.30,0.88,2015-08-02
5,Central and northern Great Barrier Reef,Australia,742.91,Sheltered,63.92,Population,2.00,297.64,0.30,0.88,2015-08-02
6,Central and northern Great Barrier Reef,Australia,38514.12,Sheltered,48.80,Population,13.75,298.86,1.43,0.84,2015-08-29
7,Central and northern Great Barrier Reef,Australia,38506.76,Sheltered,48.80,Population,22.50,298.86,1.43,0.84,2015-08-29
8,Central and northern Great Barrier Reef,Australia,35199.40,Sheltered,50.97,Population,28.75,298.87,0.83,0.88,2015-08-30
9,Central and northern Great Barrier Reef,Australia,37626.29,Sheltered,52.46,Population,3.25,298.90,0.95,0.87,2015-08-30


In [53]:
def plot_bleaching_over_time(df):
    # Ensure 'date' is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Create a line plot
    fig = px.line(df, x='date', y='percent_bleaching', 
                  title='Percent Bleaching Over Time',
                  labels={'percent_bleaching': 'Percent Bleaching', 'date': 'Date'})

    # Customize the layout for better aesthetics
    fig.update_layout(xaxis_title='Date', yaxis_title='Percent Bleaching',
                      template='plotly_dark')  # Optional: Change to plotly or other templates

    # Show the plot
    fig.show()


plot_bleaching_over_time(df_gb_cleaned)